# Notebook 6: All Three Types of Naïve Bayes

## Overview
Naïve Bayes classifiers apply **Bayes' theorem** with the **naïve assumption** that features are conditionally independent given the class.

P(y | x₁, x₂, …, xₙ) ∝ P(y) × ∏ P(xᵢ | y)

### Three Variants
| Variant | Best for | Feature Distribution |
|---|---|---|
| **Gaussian NB** | Continuous features | Gaussian (Normal) |
| **Multinomial NB** | Discrete count data (text) | Multinomial |
| **Bernoulli NB** | Binary / boolean features | Bernoulli (0/1) |

---

## Datasets Used

| Variant | Dataset | Kaggle Link |
|---|---|---|
| Gaussian NB | Heart Disease | https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset |
| Multinomial NB | SMS Spam Collection | https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset |
| Bernoulli NB | SMS Spam Collection (binarized) | same as above |

### How to Download
```bash
pip install kaggle
# Place kaggle.json at ~/.kaggle/kaggle.json

# Heart Disease dataset
kaggle datasets download -d johnsmith88/heart-disease-dataset \
  -p data/heart_disease --unzip

# SMS Spam dataset
kaggle datasets download -d uciml/sms-spam-collection-dataset \
  -p data/sms_spam --unzip
```

In [ ]:
# ─── Install (uncomment if needed) ─────────────────────────────────────────
# !pip install kaggle pandas scikit-learn matplotlib seaborn

# ─── Download datasets ──────────────────────────────────────────────────────
# !kaggle datasets download -d johnsmith88/heart-disease-dataset -p data/heart_disease --unzip
# !kaggle datasets download -d uciml/sms-spam-collection-dataset -p data/sms_spam --unzip

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, classification_report,
    ConfusionMatrixDisplay, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

---
## Part A — Gaussian Naïve Bayes
### Assumes features follow a Normal (Gaussian) distribution.
### Dataset: Heart Disease

In [ ]:
# Load Heart Disease dataset
HEART_PATH = 'data/heart_disease/heart.csv'

try:
    heart_df = pd.read_csv(HEART_PATH)
    print(f"Heart Disease dataset loaded: {heart_df.shape}")
except FileNotFoundError:
    print("Heart disease CSV not found – using synthetic data.")
    from sklearn.datasets import make_classification
    X_raw, y_raw = make_classification(
        n_samples=303, n_features=13, n_informative=8,
        n_redundant=2, random_state=42
    )
    cols = ['age','sex','cp','trestbps','chol','fbs',
            'restecg','thalach','exang','oldpeak','slope','ca','thal']
    heart_df = pd.DataFrame(X_raw, columns=cols)
    heart_df['target'] = y_raw

# Handle missing values
heart_df.fillna(heart_df.median(numeric_only=True), inplace=True)

X_h = heart_df.drop(columns=['target'])
y_h = heart_df['target']

X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    X_h, y_h, test_size=0.20, random_state=42, stratify=y_h
)

# Gaussian NB does NOT require scaling, but let's demonstrate it works either way
gnb = GaussianNB()
gnb.fit(X_h_train, y_h_train)
y_pred_g = gnb.predict(X_h_test)
y_prob_g = gnb.predict_proba(X_h_test)[:, 1]

print("\n=== Gaussian Naïve Bayes — Heart Disease ===")
print(f"Test Accuracy : {accuracy_score(y_h_test, y_pred_g):.4f}")
print(f"ROC-AUC       : {roc_auc_score(y_h_test, y_prob_g):.4f}")
print("\nClassification Report:")
print(classification_report(y_h_test, y_pred_g, target_names=['No Disease', 'Disease']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(
    y_h_test, y_pred_g,
    display_labels=['No Disease', 'Disease'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Gaussian NB — Confusion Matrix')

# Feature-wise Gaussian distributions for two classes
feature = 'age' if 'age' in X_h.columns else X_h.columns[0]
for cls, colour, label in zip([0, 1], ['steelblue', 'salmon'], ['No Disease', 'Disease']):
    subset = X_h[y_h == cls][feature]
    axes[1].hist(subset, bins=15, alpha=0.6, color=colour, label=label, density=True)
axes[1].set_title(f'Gaussian Distribution — feature: {feature}')
axes[1].set_xlabel(feature); axes[1].legend()

plt.tight_layout()
plt.show()

### Gaussian NB: Visualise class-conditional means and variances

In [ ]:
means_df = pd.DataFrame(
    gnb.theta_,              # class means
    columns=X_h.columns,
    index=['Class 0', 'Class 1']
)
print("Class-conditional feature means:")
print(means_df.T.to_string())

---
## Part B — Multinomial Naïve Bayes
### Best for discrete count data (word frequencies in text).
### Dataset: SMS Spam Collection

In [ ]:
# Possible file names in the Kaggle download
SMS_PATHS = [
    'data/sms_spam/spam.csv',
    'data/sms_spam/SMSSpamCollection',
]

sms_df = None
for path in SMS_PATHS:
    try:
        sms_df = pd.read_csv(path, encoding='latin-1')
        print(f"SMS dataset loaded from {path}: {sms_df.shape}")
        break
    except FileNotFoundError:
        continue

if sms_df is None:
    print("SMS dataset not found – creating synthetic text data.")
    import random
    random.seed(42)
    spam_msgs = [
        "Free prize winner claim now", "Win cash lottery click here",
        "Urgent prize notification claim", "Congratulations you won",
        "Click link free gift card", "You have been selected winner",
        "Cash prize winner free offer", "Limited time offer free gift",
    ]
    ham_msgs  = [
        "Hi how are you doing today", "See you at the meeting tomorrow",
        "Can we schedule a call next week", "I will be home late tonight",
        "Please review the attached document", "Let me know if you need help",
        "Dinner at eight sounds good", "Thanks for your message",
    ]
    labels = ['spam'] * 200 + ['ham'] * 800
    texts  = [random.choice(spam_msgs) for _ in range(200)] + \
             [random.choice(ham_msgs)  for _ in range(800)]
    sms_df = pd.DataFrame({'v1': labels, 'v2': texts})
    print(f"Synthetic SMS dataset created: {sms_df.shape}")

# Standardise column names
if 'label' in sms_df.columns and 'text' in sms_df.columns:
    sms_df = sms_df[['label','text']]
    sms_df.columns = ['v1', 'v2']
elif 'Category' in sms_df.columns and 'Message' in sms_df.columns:
    sms_df.rename(columns={'Category':'v1','Message':'v2'}, inplace=True)

sms_df = sms_df[['v1','v2']]
sms_df.columns = ['label', 'text']
sms_df['label_enc'] = (sms_df['label'] == 'spam').astype(int)

print("\nLabel distribution:")
print(sms_df['label'].value_counts())
sms_df.head()

In [ ]:
# Vectorise: convert text to word-count matrix
vectoriser = CountVectorizer(stop_words='english', max_features=5000)

X_sms = sms_df['text']
y_sms = sms_df['label_enc']

X_sms_tr, X_sms_te, y_sms_tr, y_sms_te = train_test_split(
    X_sms, y_sms, test_size=0.20, random_state=42, stratify=y_sms
)

# Fit vectoriser on training set only
X_sms_tr_vec = vectoriser.fit_transform(X_sms_tr)
X_sms_te_vec = vectoriser.transform(X_sms_te)

print(f"Vocabulary size : {len(vectoriser.vocabulary_)}")
print(f"Feature matrix  : {X_sms_tr_vec.shape}")

# Multinomial NB
mnb = MultinomialNB(alpha=1.0)    # alpha is Laplace smoothing
mnb.fit(X_sms_tr_vec, y_sms_tr)
y_pred_m = mnb.predict(X_sms_te_vec)

print("\n=== Multinomial Naïve Bayes — SMS Spam ===")
print(f"Test Accuracy : {accuracy_score(y_sms_te, y_pred_m):.4f}")
print(f"ROC-AUC       : {roc_auc_score(y_sms_te, mnb.predict_proba(X_sms_te_vec)[:, 1]):.4f}")
print("\nClassification Report:")
print(classification_report(y_sms_te, y_pred_m, target_names=['Ham', 'Spam']))

In [ ]:
# Top words for spam and ham
feature_names = vectoriser.get_feature_names_out()
top_n = 15

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for cls_idx, (cls_name, ax) in enumerate(zip(['Ham', 'Spam'], axes)):
    log_prob = mnb.feature_log_prob_[cls_idx]
    top_idx  = np.argsort(log_prob)[-top_n:][::-1]
    ax.barh(feature_names[top_idx][::-1], log_prob[top_idx][::-1], color='steelblue')
    ax.set_title(f'Top {top_n} {cls_name} words (log-prob)')
    ax.set_xlabel('Log probability')

plt.tight_layout()
plt.show()

---
## Part C — Bernoulli Naïve Bayes
### Designed for binary features (word present/absent).
### Dataset: SMS Spam Collection (binarized)

In [ ]:
# Binarize: 1 if word appears in message, 0 otherwise
bin_vectoriser = CountVectorizer(
    stop_words='english',
    max_features=5000,
    binary=True   # ← key difference from Multinomial
)

X_sms_tr_bin = bin_vectoriser.fit_transform(X_sms_tr)
X_sms_te_bin = bin_vectoriser.transform(X_sms_te)

bnb = BernoulliNB(alpha=1.0)
bnb.fit(X_sms_tr_bin, y_sms_tr)
y_pred_b = bnb.predict(X_sms_te_bin)

print("=== Bernoulli Naïve Bayes — SMS Spam ===")
print(f"Test Accuracy : {accuracy_score(y_sms_te, y_pred_b):.4f}")
print(f"ROC-AUC       : {roc_auc_score(y_sms_te, bnb.predict_proba(X_sms_te_bin)[:, 1]):.4f}")
print("\nClassification Report:")
print(classification_report(y_sms_te, y_pred_b, target_names=['Ham', 'Spam']))

---
## Comparison of All Three Naïve Bayes Classifiers

In [ ]:
comparison = pd.DataFrame([
    {
        'Model': 'Gaussian NB',
        'Dataset': 'Heart Disease',
        'Feature Type': 'Continuous',
        'Accuracy': accuracy_score(y_h_test, y_pred_g),
        'ROC-AUC': roc_auc_score(y_h_test, y_prob_g)
    },
    {
        'Model': 'Multinomial NB',
        'Dataset': 'SMS Spam (word counts)',
        'Feature Type': 'Count (int ≥ 0)',
        'Accuracy': accuracy_score(y_sms_te, y_pred_m),
        'ROC-AUC': roc_auc_score(y_sms_te, mnb.predict_proba(X_sms_te_vec)[:, 1])
    },
    {
        'Model': 'Bernoulli NB',
        'Dataset': 'SMS Spam (binary)',
        'Feature Type': 'Binary (0/1)',
        'Accuracy': accuracy_score(y_sms_te, y_pred_b),
        'ROC-AUC': roc_auc_score(y_sms_te, bnb.predict_proba(X_sms_te_bin)[:, 1])
    },
])

comparison['Accuracy'] = comparison['Accuracy'].map('{:.4f}'.format)
comparison['ROC-AUC']  = comparison['ROC-AUC'].map('{:.4f}'.format)
print(comparison.to_string(index=False))

## Summary

| Variant | Feature Type | Key Parameter |
|---|---|---|
| **Gaussian NB** | Continuous real-valued | `var_smoothing` |
| **Multinomial NB** | Discrete counts (text, TF) | `alpha` (Laplace smoothing) |
| **Bernoulli NB** | Binary presence/absence | `alpha`, `binarize` |

* All three variants are **fast**, **simple**, and work well even with small datasets.
* They are excellent **baselines** before trying more complex models.
* The naïve independence assumption rarely holds perfectly, but the classifiers are surprisingly robust.